In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/miheeranikam/renderserver/render_server.py


In [3]:
%pip install -q "fastapi>=0.115" "uvicorn[standard]" python-multipart diffusers transformers accelerate safetensors bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import getpass
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

os.environ["REMOTE_RENDER_TOKEN"] = getpass.getpass("RoomSage bearer token: ")

Torch: 2.11.0+cu130
CUDA: 13.0
GPU available: True
Device: Tesla T4


RoomSage bearer token:  ········


In [9]:
from pathlib import Path
import shutil

matches = list(Path("/kaggle/input").rglob("render_server.py"))
assert matches, "render_server.py was not found. Attach the dataset with Add Input."

src = matches[0]
dst = Path("/kaggle/working/render_server.py")
shutil.copy2(src, dst)

text = dst.read_text()
text = text.replace("use_safetensors=True", "use_safetensors=False")
text = text.replace('variant="fp16"', 'variant="fp16"')
dst.write_text(text)

print("Using:", dst)

Using: /kaggle/working/render_server.py


In [10]:
import subprocess
import sys
import time
import urllib.request

subprocess.run(["pkill", "-f", "render_server.py"], check=False)

server_log = open("/tmp/roomsage-server.log", "w")

server = subprocess.Popen(
    [sys.executable, "/kaggle/working/render_server.py"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

for _ in range(60):
    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:8000/health",
            timeout=3,
        )
        print(response.read().decode())
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError(open("/tmp/roomsage-server.log").read())
        time.sleep(1)
else:
    raise TimeoutError("RoomSage server did not start")

{"status":"ok","model_id":"stable-diffusion-v1-5/stable-diffusion-inpainting","mode":"inpaint","weight_variant":"fp16","safetensors":"true","license":"CreativeML OpenRAIL-M"}


In [16]:
!pkill -f cloudflared

In [17]:
import os
import re
import subprocess
import time
import urllib.request

cloudflared = "/kaggle/working/cloudflared"

urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    cloudflared,
)
os.chmod(cloudflared, 0o755)

tunnel_log = open("/tmp/cloudflared.log", "w")

tunnel = subprocess.Popen(
    [cloudflared, "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

tunnel_url = None

for _ in range(60):
    tunnel_log.flush()
    log_text = open(
        "/tmp/cloudflared.log",
        encoding="utf-8",
        errors="replace",
    ).read()

    match = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com",
        log_text,
    )

    if match:
        tunnel_url = match.group(0)
        break

    if tunnel.poll() is not None:
        raise RuntimeError(log_text)

    time.sleep(1)

if not tunnel_url:
    raise TimeoutError("Cloudflare tunnel URL was not found")

print("REMOTE_RENDER_URL:", tunnel_url + "/render")
print("Health URL:", tunnel_url + "/health")

REMOTE_RENDER_URL: https://hardwood-harmony-half-males.trycloudflare.com/render
Health URL: https://hardwood-harmony-half-males.trycloudflare.com/health


In [18]:
print(
    urllib.request.urlopen(
        tunnel_url + "/health",
        timeout=20,
    ).read().decode()
)

{"status":"ok","model_id":"stable-diffusion-v1-5/stable-diffusion-inpainting","mode":"inpaint","weight_variant":"fp16","safetensors":"true","license":"CreativeML OpenRAIL-M"}


In [13]:
!pip install -q fastapi uvicorn python-multipart diffusers transformers accelerate safetensors bitsandbytes

In [ ]:
import os, getpass
os.environ["REMOTE_RENDER_TOKEN"] = getpass.getpass("RoomSage bearer token: ")

In [ ]:
from pathlib import Path
import os, subprocess, sys, time, urllib.request
matches = list(Path("/kaggle/input").rglob("render_server.py"))
if not matches:
    raise FileNotFoundError("Add the dataset containing render_server.py with Kaggle's Add Input button")
script = matches[0]
print("Using server file:", script)
server_log = open("/tmp/roomsage-server.log", "w")
server = subprocess.Popen(
       [sys.executable, str(script)],
       stdout=server_log,
       stderr=subprocess.STDOUT,
       start_new_session=True,
   )
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2).read()
        print("RoomSage server is ready")
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError(open("/tmp/roomsage-server.log").read())
        time.sleep(1)
    else:
        raise TimeoutError("RoomSage server did not become ready; inspect /tmp/roomsage-server.log")

In [ ]:
import os
import subprocess

# Kill any active cloudflared process
subprocess.run(["pkill", "-f", "cloudflared"])

# Remove the locked file if it exists
cloudflared = os.path.abspath("./cloudflared")
if os.path.exists(cloudflared):
    os.remove(cloudflared)

In [ ]:
import os, re, subprocess, time, urllib.request
cloudflared = os.path.abspath("./cloudflared")
urllib.request.urlretrieve(
       "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
       cloudflared,
   )
os.chmod(cloudflared, 0o755)
tunnel_log = open("/tmp/cloudflared.log", "w")
tunnel = subprocess.Popen(
       [cloudflared, "tunnel", "--url", "http://127.0.0.1:8000"],
       stdout=tunnel_log,
       stderr=subprocess.STDOUT,
       start_new_session=True,
   )
tunnel_url = None
for _ in range(60):
    tunnel_log.flush()
    try:
        log_text = open("/tmp/cloudflared.log", encoding="utf-8", errors="replace").read()
    except FileNotFoundError:
        log_text = ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError("cloudflared exited; log:\n" + log_text)
    time.sleep(1)
if not tunnel_url:
    raise TimeoutError("No trycloudflare URL appeared; inspect /tmp/cloudflared.log")
print("Tunnel URL:", tunnel_url)

In [19]:
os.environ["REMOTE_RENDER_URL"] = "https://hardwood-harmony-half-males.trycloudflare.com/render" 

In [ ]:
print(open("/tmp/roomsage-server.log", encoding="utf-8", errors="replace").read()[-5000:])

In [ ]:
print(open("/tmp/roomsage-server.log", encoding="utf-8", errors="replace").read()[-8000:])